# Prepare 128×128 Isotropic turbulence tiles

This notebook converts the original 512×512 isotropic turbulence snapshots
stored in `data/raw/isotropic_xy_512/isotropic_xy_512.h5` into a new dataset
of 128×128 tiles.

Can be used to train a lightweight SR model.

In [2]:
import sys
import os
from pathlib import Path

import h5py
import numpy as np
from tqdm.notebook import tqdm


def find_project_root(start: Path, target: str = "models", max_up: int = 8) -> Path:
    root = start.resolve()
    for _ in range(max_up):
        if (root / target).exists():
            return root
        if root.parent == root:
            break
        root = root.parent
    raise FileNotFoundError(f"Could not find '{target}' above {start}")


# ---------------------------------------------------------------------------
# Locate project root and input/output paths
# ---------------------------------------------------------------------------
ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root added to sys.path:", ROOT)

SRC_PATH = ROOT / "data" / "raw" / "isotropic_xy_512" / "isotropic_xy_512.h5"
DST_DIR = ROOT / "data" / "raw" / "isotropic_xy_128"
DST_PATH = DST_DIR / "isotropic_xy_128.h5"

print(f"Project root: {ROOT}")
print(f"Source HDF5: {SRC_PATH}")
print(f"Destination directory: {DST_DIR}")
print(f"Destination HDF5: {DST_PATH}\n")

if not SRC_PATH.exists():
    raise FileNotFoundError(f"Source file not found: {SRC_PATH}")

DST_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Define tiling pattern: 8 × 128×128 tiles per 512×512 snapshot
# ---------------------------------------------------------------------------
TILE_SIZE = 128
FIELD_SIZE = 512

if FIELD_SIZE % TILE_SIZE != 0:
    raise ValueError("TILE_SIZE must divide FIELD_SIZE exactly.")

tiles_per_dim = FIELD_SIZE // TILE_SIZE  # 4 for 512→128

# Build a 4×4 grid and keep every other tile (checkerboard) → 8 tiles
tile_positions = []  # list of (row_start, col_start)
for i in range(tiles_per_dim):
    for j in range(tiles_per_dim):
        if (i + j) % 2 == 0:
            row = i * TILE_SIZE
            col = j * TILE_SIZE
            tile_positions.append((row, col))

print("Tiling configuration:")
print(f"  Field size: {FIELD_SIZE}×{FIELD_SIZE}")
print(f"  Tile size:  {TILE_SIZE}×{TILE_SIZE}")
print(f"  Grid:       {tiles_per_dim}×{tiles_per_dim}")
print(f"  Tiles kept: {len(tile_positions)} per snapshot\n")

# ---------------------------------------------------------------------------
# Create the 128×128 HDF5 dataset
# ---------------------------------------------------------------------------
if DST_PATH.exists():
    print(f"⚠️  Destination file already exists and will be overwritten: {DST_PATH}")
    DST_PATH.unlink()

with h5py.File(SRC_PATH, "r") as f_src, h5py.File(DST_PATH, "w") as f_dst:
    src_snaps = f_src["snapshots"]
    dst_snaps = f_dst.create_group("snapshots")

    snap_keys = sorted(src_snaps.keys())
    num_snaps = len(snap_keys)
    tiles_per_snapshot = len(tile_positions)
    total_tiles = num_snaps * tiles_per_snapshot

    print(f"Found {num_snaps} snapshots in source file.")
    print(f"Tiles per snapshot: {tiles_per_snapshot}")
    print(f"Total output tiles: {total_tiles}\n")

    # Sanity check on field shapes using the first snapshot
    first = src_snaps[snap_keys[0]]
    omega_shape = first["omega_z"].shape
    if omega_shape != (FIELD_SIZE, FIELD_SIZE):
        raise ValueError(
            f"Expected omega_z shape {(FIELD_SIZE, FIELD_SIZE)}, got {omega_shape}"
        )

    tile_index = 0

    for snap_idx, snap_key in enumerate(tqdm(snap_keys, desc="Tiling snapshots")):
        g_src = src_snaps[snap_key]
        omega = g_src["omega_z"][...]
        ux = g_src["ux"][...]
        uy = g_src["uy"][...]
        uz = g_src["uz"][...]
        time_val = g_src["time"][()]

        for (row0, col0) in tile_positions:
            row1 = row0 + TILE_SIZE
            col1 = col0 + TILE_SIZE

            omega_tile = omega[row0:row1, col0:col1]
            ux_tile = ux[row0:row1, col0:col1]
            uy_tile = uy[row0:row1, col0:col1]
            uz_tile = uz[row0:row1, col0:col1]

            grp_name = f"{tile_index:04d}"
            g_dst = dst_snaps.create_group(grp_name)

            g_dst.create_dataset("omega_z", data=omega_tile, compression="gzip")
            g_dst.create_dataset("ux", data=ux_tile, compression="gzip")
            g_dst.create_dataset("uy", data=uy_tile, compression="gzip")
            g_dst.create_dataset("uz", data=uz_tile, compression="gzip")
            g_dst.create_dataset("time", data=time_val)

            tile_index += 1

print("\n✓ Finished writing 128×128 isotropic tiles.")
print(f"  Output file: {DST_PATH}")
print(f"  Total tiles: {total_tiles}")


Project root added to sys.path: /workspaces/jhtdb-ml
Project root: /workspaces/jhtdb-ml
Source HDF5: /workspaces/jhtdb-ml/data/raw/isotropic_xy_512/isotropic_xy_512.h5
Destination directory: /workspaces/jhtdb-ml/data/raw/isotropic_xy_128
Destination HDF5: /workspaces/jhtdb-ml/data/raw/isotropic_xy_128/isotropic_xy_128.h5

Tiling configuration:
  Field size: 512×512
  Tile size:  128×128
  Grid:       4×4
  Tiles kept: 8 per snapshot

Found 999 snapshots in source file.
Tiles per snapshot: 8
Total output tiles: 7992



Tiling snapshots:   0%|          | 0/999 [00:00<?, ?it/s]


✓ Finished writing 128×128 isotropic tiles.
  Output file: /workspaces/jhtdb-ml/data/raw/isotropic_xy_128/isotropic_xy_128.h5
  Total tiles: 7992
